In [229]:
## load packages 
import pandas as pd
import re
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

## nltk imports
from nltk.tokenize import word_tokenize, wordpunct_tokenize
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

## sklearn imports
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

## print mult things
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## random
import random

In [230]:
#defining functions

#defining function to create tract_num data that extracts the tract number from a dataset's census_tract column and converts into float
def add_tract_num(df):
    df['tract_num'] = (
        df['census_tract']
        .str.extract(r'Census Tract (\d+\.?\d*)')[0]
        .astype(float)
        .round(2)
    )
    return df

#manually mapping tracts that aren't correctly aggregated by the function below
manual_map = {
    41.03: 41.01,
    41.04: 41.01,
    150.03: 150.02,
    150.04: 150.02,
}

#defining function to aggregate tracts so that they match 2010 tracts
def get_match_key(tract):
    tract = float(tract)
    if tract in manual_map: #applying manual map first
        return manual_map[tract]
    if tract in tracts_2010: #if the tract is the same in 2020 as it was in 2010, don't change it
        return tract
    integer_parent = float(int(tract))
    if integer_parent in tracts_2010: #otherwise, if its integer parent was in 2010, return the integer parent
        return integer_parent
    return tract         

#defining function that applies the get_match_key function to the 2020 race dataset
def aggregate_race(df, count_cols, bool_cols=None):
    df['match_key'] = df['tract_num'].apply(get_match_key)

    #aggregating value columns
    for col in count_cols:
        df[col] = df[col].astype(str).str.replace(',', '').pipe(pd.to_numeric, errors='coerce')
    df[count_cols] = df[count_cols].apply(pd.to_numeric, errors='coerce') #using apply here won't impact runtime because it is only running through a very short list
    
    df_agg = df.groupby('match_key')[count_cols].sum().reset_index()
    df_agg.rename(columns={'match_key': 'tract_num'}, inplace=True)

    #aggregating boolean columns. If either 2020 child tract of a 2010 parent tract was true, mark the parent tract as true
    if bool_cols:
        bool_agg = df.groupby('match_key')[bool_cols].any().reset_index()
        bool_agg.rename(columns={'match_key': 'tract_num'}, inplace=True)
        df_agg = df_agg.merge(bool_agg, on='tract_num')
    
    return df_agg

#defining function that applies the get_match_key function to the ACS datasets, using household-weighted averages of medians when combining tracts
def aggregate_ACS(df, value_col, weight_col='household_num'):
    df['match_key'] = df['tract_num'].apply(get_match_key)
    df['weighted'] = df[value_col] * df[weight_col] #weighting the median
    df_agg = df.groupby('match_key').agg( 
        weighted_sum=('weighted', 'sum'),
        total_weight=(weight_col, 'sum')
    ).reset_index()
    df_agg[value_col] = df_agg['weighted_sum'] / df_agg['total_weight']
    df_agg = df_agg[['match_key', value_col]].rename(columns={'match_key': 'tract_num'})
    return df_agg

#defining function to create the train_access column after merging. The function assigns one of four values based on a tract's values in the near_hblr and near_path columns
def categorize(row):
    if row['near_hblr_x'] and not row['near_path_x']:
        return 'hblr_only'
    elif row['near_path_x'] and not row['near_hblr_x']:
        return 'path_only'
    elif row['near_hblr_x'] and row['near_path_x']:
        return 'near_both'
    else:
        return 'near_neither'

In [231]:
#pulling data from previous file
df2020 = pd.read_csv('../data/2020cleaned1.csv')
df2010 = pd.read_csv('../data/2010cleaned1.csv')
df20hs = pd.read_csv('../data/2020housing.csv')
df10hs = pd.read_csv('../data/2010housing.csv')
df20rent = pd.read_csv('../data/2020rent.csv')
df10rent = pd.read_csv('../data/2010rent.csv')
df20inc = pd.read_csv('../data/2020income.csv')
df10inc = pd.read_csv('../data/2010income.csv')

In [232]:
#coding new tract_num column to standardize the numbers of the tracts in each dataset in order to permit merging
df2010['tract_num'] = df2010['Label (Grouping)'].str.extract(r'(\d+\.?\d*)')[0].astype(float)
tracts_2010 = set(df2010['tract_num'])
df2020['tract_num'] = df2020['census_tract'].str.extract(r'(\d+\.?\d*)')[0].astype(float)

#for the 
dfs = {'df10rent': df10rent, 'df20rent': df20rent, 'df10hs': df10hs, 'df20hs': df20hs, 'df20inc': df20inc, 'df10inc': df10inc}
for name, df in dfs.items():
    dfs[name] = add_tract_num(df)
df10rent, df20rent, df10hs, df20hs, df20inc, df10inc = dfs.values()

In [233]:
df2020.head()

,census_tract,total_pop,under_5,5_to_9,10 to 14 years,15 to 19 years,20 to 24 years,25 to 29 years,30 to 34 years,35 to 39 years,...,total_asian,total_other,total_hispanic,white_pct,black_pct,asian_pct,other_pct,hispanic_pct,near_path,tract_num
0,Census Tract 1.01; Hudson County; New Jersey!!...,2554.0,160.0,161.0,171.0,146.0,173.0,265,247,213,...,864.0,35.0,1059.0,17.815192,4.111198,33.829287,1.370399,41.464370,False,1.01
1,Census Tract 1.02; Hudson County; New Jersey!!...,3834.0,211.0,205.0,215.0,211.0,271.0,350,426,322,...,1233.0,31.0,1590.0,18.309859,5.685968,32.159624,0.808555,41.471049,False,1.02
2,Census Tract 2; Hudson County; New Jersey!!Count,5391.0,273.0,345.0,301.0,282.0,393.0,519,561,437,...,1043.0,86.0,2954.0,15.933964,5.935819,19.347060,1.595251,54.795029,False,2.00
3,Census Tract 3; Hudson County; New Jersey!!Count,3948.0,198.0,215.0,201.0,185.0,292.0,476,420,384,...,496.0,42.0,1910.0,31.636272,3.571429,12.563323,1.063830,48.378926,False,3.00
4,Census Tract 4; Hudson County; New Jersey!!Count,3973.0,247.0,258.0,205.0,196.0,280.0,444,376,311,...,1654.0,46.0,1372.0,16.763151,3.674805,41.631009,1.157815,34.533098,False,4.00


In [234]:
df10rent.head()
df20inc.head()

,census_tract,Median gross rent,tract_num,median_rent_2010
0,"Census Tract 58.02, Hudson County, New Jersey!...","2,000+",58.02,2000
1,"Census Tract 74, Hudson County, New Jersey!!To...","2,000+",74.00,2000
2,"Census Tract 76, Hudson County, New Jersey!!To...","2,000+",76.00,2000
3,"Census Tract 77, Hudson County, New Jersey!!To...","2,000+",77.00,2000
4,"Census Tract 150.01, Hudson County, New Jersey...","2,000+",150.01,2000


,0,Geography,census_tract,household_num,Estimate!!Median income (dollars)!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households,median_income_2020,tract_num
0,1,1400000US34017000101,"Census Tract 1.01, Hudson County, New Jersey",848,91300,91300.0,1.01
1,2,1400000US34017000102,"Census Tract 1.02, Hudson County, New Jersey",1135,57841,57841.0,1.02
2,3,1400000US34017000200,"Census Tract 2, Hudson County, New Jersey",1933,39665,39665.0,2.00
3,4,1400000US34017000300,"Census Tract 3, Hudson County, New Jersey",1477,61069,61069.0,3.00
4,5,1400000US34017000400,"Census Tract 4, Hudson County, New Jersey",1246,80324,80324.0,4.00


In [235]:
#aggregating tracts for 2020 race
df2020_agg = aggregate_race(
    df2020,
    count_cols=['total_pop', 'total_white', 'total_black', 'total_asian', 'total_hispanic', 'total_other'],
    bool_cols=['near_hblr', 'near_path']
)

In [236]:
df2020_agg.head()

,tract_num,total_pop,total_white,total_black,total_asian,total_hispanic,total_other,near_hblr,near_path
0,1.0,6388.0,1157.0,323.0,2097.0,2649.0,66.0,False,False
1,2.0,5391.0,859.0,320.0,1043.0,2954.0,86.0,False,False
2,3.0,3948.0,1249.0,141.0,496.0,1910.0,42.0,True,False
3,4.0,3973.0,666.0,146.0,1654.0,1372.0,46.0,False,False
4,5.0,4317.0,957.0,197.0,1310.0,1677.0,58.0,False,False


In [237]:
#creating racial proportion columns again using aggregated totals for each tract
for race in ['total_white', 'total_black', 'total_asian', 'total_hispanic', 'total_other']:
    df2020_agg[f'{race}_pct'] = df2020_agg[race] / df2020_agg['total_pop'] * 100


In [238]:
df2020_agg.head()

,tract_num,total_pop,total_white,total_black,total_asian,total_hispanic,total_other,near_hblr,near_path,total_white_pct,total_black_pct,total_asian_pct,total_hispanic_pct,total_other_pct
0,1.0,6388.0,1157.0,323.0,2097.0,2649.0,66.0,False,False,18.112085,5.056356,32.827176,41.468378,1.033187
1,2.0,5391.0,859.0,320.0,1043.0,2954.0,86.0,False,False,15.933964,5.935819,19.347060,54.795029,1.595251
2,3.0,3948.0,1249.0,141.0,496.0,1910.0,42.0,True,False,31.636272,3.571429,12.563323,48.378926,1.063830
3,4.0,3973.0,666.0,146.0,1654.0,1372.0,46.0,False,False,16.763151,3.674805,41.631009,34.533098,1.157815
4,5.0,4317.0,957.0,197.0,1310.0,1677.0,58.0,False,False,22.168172,4.563354,30.345147,38.846421,1.343526


In [239]:
#identifying tracts with null household num values
problem_tracts = df20inc[df20inc['household_num'].isna() | (df20inc['household_num'] == 0)]
print(problem_tracts[['tract_num', 'median_income_2020', 'household_num']])

     tract_num  median_income_2020  household_num
182     9801.0                 NaN              0


In [240]:
#dropping tract 9801 because it is not a populated tract
df20inc = df20inc.dropna(subset=['household_num', 'median_income_2020'])
df20inc = df20inc[df20inc['household_num'] > 0]

In [241]:
# Aggregating 2020 income tracts using household-weighted average of medians
df20incagg = aggregate_ACS(df20inc, 'median_income_2020')
df20incagg.head()

,tract_num,median_income_2020
0,1.0,72149.236006
1,2.0,39665.000000
2,3.0,61069.000000
3,4.0,80324.000000
4,5.0,76406.000000


In [242]:
#Printing diagnostics before merge
print("=== BEFORE MERGES ===")
print(f"df20hs shape: {df20hs.shape}")
print(f"df20rent shape: {df20rent.shape}")
print(f"df20inc tracts available: {df20inc['tract_num'].nunique()}")
print(f"df20hs tracts: {df20hs['tract_num'].nunique()}")
print(f"df20rent tracts: {df20rent['tract_num'].nunique()}")

=== BEFORE MERGES ===
df20hs shape: (183, 4)
df20rent shape: (183, 4)
df20inc tracts available: 180
df20hs tracts: 183
df20rent tracts: 183


In [243]:
#merging household_num column from 2020 income data into 2020 house and rent data in order to allow aggregation using household-weighted average of medians
df20hs = df20hs.merge(
    df20inc[['tract_num', 'household_num']], 
    on='tract_num', 
    how='left'
)
df20rent = df20rent.merge(
    df20inc[['tract_num', 'household_num']], 
    on='tract_num', 
    how='left'
)

In [244]:
#Printing diagnostics after merge
print("\n=== AFTER MERGES ===")
print(f"df20hs shape: {df20hs.shape}")
print(f"df20rent shape: {df20rent.shape}")
print(f"df20hs household_num nulls: {df20hs['household_num'].isna().sum()}")
print(f"df20rent household_num nulls: {df20rent['household_num'].isna().sum()}")


=== AFTER MERGES ===
df20hs shape: (183, 5)
df20rent shape: (183, 5)
df20hs household_num nulls: 3
df20rent household_num nulls: 3


In [245]:
# Aggregating 2020 rent price tracts using household-weighted average of medians
df20rentagg = aggregate_ACS(df20rent, 'median_rent_2020')
df20rentagg.head()

,tract_num,median_rent_2020
0,1.0,1545.67171
1,2.0,1314.00000
2,3.0,1288.00000
3,4.0,1492.00000
4,5.0,1534.00000


In [246]:
# Aggregating 2020 house price tracts using household-weighted average of medians
df20hsagg = aggregate_ACS(df20hs, 'median_house_2020')
df20hsagg.head()

,tract_num,median_house_2020
0,1.0,406290.872416
1,2.0,467700.000000
2,3.0,538400.000000
3,4.0,421200.000000
4,5.0,515500.000000


In [247]:
#Before merge diagnostics
print("=== BEFORE MERGE ===")
for name, df in [('df2010', df2010), ('df2020_agg', df2020_agg), ('df20hsagg', df20hsagg), 
                  ('df20incagg', df20incagg), ('df20rentagg', df20rentagg), 
                  ('df10hs', df10hs), ('df10inc', df10inc), ('df10rent', df10rent)]:
    print(f"{name}: {df.shape} | tracts: {df['tract_num'].nunique()}")

#Merging all cleaned datasets
dfmerged = df2010
for df in [df2020_agg, df20hsagg, df20incagg, df20rentagg, df10hs, df10inc, df10rent]:
    dfmerged = dfmerged.merge(df, on='tract_num', how='left')

#After merge diagnostics
print("\n=== AFTER MERGE ===")
print(f"merged shape: {dfmerged.shape}")
print(f"tracts: {dfmerged['tract_num'].nunique()}")
print(f"nulls per column:\n{dfmerged.isnull().sum()}")

=== BEFORE MERGE ===
df2010: (166, 24) | tracts: 166
df2020_agg: (166, 14) | tracts: 166
df20hsagg: (166, 2) | tracts: 166
df20incagg: (163, 2) | tracts: 163
df20rentagg: (166, 2) | tracts: 166
df10hs: (166, 4) | tracts: 166
df10inc: (166, 4) | tracts: 166
df10rent: (162, 4) | tracts: 162

=== AFTER MERGE ===
merged shape: (166, 49)
tracts: 166
nulls per column:
Label (Grouping)                                    0
Total:                                              0
Hispanic or Latino                                  0
White alone                                         0
Black or African American alone                     0
American Indian and Alaska Native alone             0
Asian alone                                         0
Native Hawaiian and Other Pacific Islander alone    0
Some Other Race alone                               0
Two or More Races:                                  0
total_pop_x                                         3
total_white_x                            

In [248]:
dfmerged.head()

,Label (Grouping),Total:,Hispanic or Latino,White alone,Black or African American alone,American Indian and Alaska Native alone,Asian alone,Native Hawaiian and Other Pacific Islander alone,Some Other Race alone,Two or More Races:,...,median_rent_2020,census_tract_x,Median value (dollars),median_house_2010,census_tract_y,Households,median_income_2010,census_tract,Median gross rent,median_rent_2010
0,"Census Tract 1, Hudson County, New Jersey","6,025","2,458","1,437",253,15,"1,693",9,32,128,...,1545.67171,"Census Tract 1, Hudson County, New Jersey!!Est...","342,900",342900.0,"Census Tract 1, Hudson County, New Jersey!!Med...","56,389",56389.0,"Census Tract 1, Hudson County, New Jersey!!Tot...","1,341",1341.0
1,"Census Tract 2, Hudson County, New Jersey","5,409","3,362",987,310,12,630,0,31,77,...,1314.00000,"Census Tract 2, Hudson County, New Jersey!!Est...","275,500",275500.0,"Census Tract 2, Hudson County, New Jersey!!Med...","38,438",38438.0,"Census Tract 2, Hudson County, New Jersey!!Tot...",962,962.0
2,"Census Tract 3, Hudson County, New Jersey","4,220 (r46394)","2,346","1,267",177,10,312,0,35,73,...,1288.00000,"Census Tract 3, Hudson County, New Jersey!!Est...","409,300",409300.0,"Census Tract 3, Hudson County, New Jersey!!Med...","46,033",46033.0,"Census Tract 3, Hudson County, New Jersey!!Tot...","1,120",1120.0
3,"Census Tract 4, Hudson County, New Jersey","3,991","1,297","1,013",205,10,"1,297",0,39,130,...,1492.00000,"Census Tract 4, Hudson County, New Jersey!!Est...","366,000",366000.0,"Census Tract 4, Hudson County, New Jersey!!Med...","70,586",70586.0,"Census Tract 4, Hudson County, New Jersey!!Tot...","1,317",1317.0
4,"Census Tract 5, Hudson County, New Jersey","4,311 (r46395)","1,928",977,224,10,"1,062",0,48,62,...,1534.00000,"Census Tract 5, Hudson County, New Jersey!!Est...","413,200",413200.0,"Census Tract 5, Hudson County, New Jersey!!Med...","52,147",52147.0,"Census Tract 5, Hudson County, New Jersey!!Tot...",963,963.0


In [249]:
#cleaning merged dataset by dropping unnecessary leftover string value columns from original data pull
dfmerged = dfmerged.drop(columns=[col for col in dfmerged.columns if re.search(r'Median ', col)])

In [250]:
#Checking some columns to ensure clean merge
dfmerged[['median_rent_2020', 'near_hblr_x', 'tract_num', 'median_house_2010', 'white_pct']].head()

,median_rent_2020,near_hblr_x,tract_num,median_house_2010,white_pct
0,1545.67171,False,1.0,342900.0,23.850622
1,1314.00000,False,2.0,275500.0,18.247366
2,1288.00000,True,3.0,409300.0,30.023697
3,1492.00000,False,4.0,366000.0,25.382110
4,1534.00000,False,5.0,413200.0,22.662955


In [251]:
#creating new four-way train_access column based on near_path and near_hblr boolean columns
dfmerged['train_access'] = dfmerged.apply(categorize, axis=1)

In [255]:
#creating percent change columns from 2010 to 2020
race_cols = ['white_pct', 'black_pct', 'asian_pct', 'hispanic_pct', 'other_pct']

for col in race_cols:
    dfmerged[f'{col}_change'] = dfmerged[f'total_{col}'] - dfmerged[f'{col}'] #the names for the racial percentages from the 2020 and 2010 datasets are slightly different, so they did not have the year suffixes attached. The columns starting with "total_" are from the 2020 dataset.
                                                                                                                     #this is an artifact of recalculating the percentages for the 2020 dataset after aggregating.

change_cols = ['census_tract'] + [f'{col}_change' for col in race_cols]

In [253]:
#observing percent change
dfmerged.groupby(['train_access'])[['white_pct_change', 'black_pct_change', 'asian_pct_change', 'hispanic_pct_change', 'other_pct_change']].mean().reset_index()

,train_access,white_pct_change,black_pct_change,asian_pct_change,hispanic_pct_change,other_pct_change
0,hblr_only,-2.591044,-3.375403,2.366953,1.903513,0.465093
1,near_both,-2.304790,-0.824907,1.501595,0.089489,0.431920
2,near_neither,-3.821287,-0.819927,2.368653,0.298351,0.725507
3,path_only,-0.935575,-1.062751,3.718139,-2.883944,0.259766


In [254]:
#Exporting merged data
dfmerged.to_csv('../data/hudsonmerged.csv', index=False)